In [1]:
import numpy as np
import pandas as pd

# Create a deterministic custom dataset: 24 months, 3 regions, 2 products.
rows = []
regions = ["North", "South", "West"]
products = ["Basic", "Premium"]

for month_index, date in enumerate(pd.date_range("2024-01-01", periods=24, freq="MS"), start=1):
    seasonality = 1 + 0.12 * np.sin(2 * np.pi * (date.month - 1) / 12)

    for region_index, region in enumerate(regions):
        for product_index, product in enumerate(products):
            units = int(
                (75 + 3 * month_index + 12 * region_index + 18 * product_index)
                * seasonality
            )
            price = 30 + 15 * product_index
            unit_cost = price * (0.55 - 0.03 * product_index)
            discount = 0.05 + 0.01 * region_index
            marketing_spend = 180 + 20 * region_index + 35 * product_index + 4 * month_index

            revenue = units * price * (1 - discount)
            profit = revenue - units * unit_cost - marketing_spend

            rows.append({
                "date": date,
                "month_index": month_index,
                "region": region,
                "product": product,
                "units": units,
                "price": price,
                "discount": discount,
                "marketing_spend": marketing_spend,
                "revenue": revenue,
                "profit": profit,
            })

df = pd.DataFrame(rows)

# 1. Descriptive analysis: overall performance
print("DESCRIPTIVE ANALYSIS")
print(df[["units", "revenue", "profit", "marketing_spend"]].describe().round(2))
print("\nRevenue and profit by region:")
print(
    df.groupby("region")[["revenue", "profit"]]
      .sum()
      .sort_values("revenue", ascending=False)
      .round(2)
)

# 2. Diagnostic analysis: compare segments and examine correlations
print("\nDIAGNOSTIC ANALYSIS")
segment_results = (
    df.groupby(["region", "product"])
      .agg(
          revenue=("revenue", "sum"),
          profit=("profit", "sum"),
          marketing_spend=("marketing_spend", "sum"),
          units=("units", "sum"),
      )
)
segment_results["profit_margin"] = segment_results["profit"] / segment_results["revenue"]
segment_results["marketing_roas"] = (
    segment_results["revenue"] / segment_results["marketing_spend"]
)
print(segment_results.round(2))

print("\nNumeric correlations:")
print(df[["units", "discount", "marketing_spend", "revenue", "profit"]]
      .corr()
      .round(2))

# 3. Predictive analysis: simple linear trend forecast for next month's revenue
monthly_revenue = (
    df.groupby("month_index")["revenue"]
      .sum()
      .sort_index()
)
slope, intercept = np.polyfit(monthly_revenue.index, monthly_revenue.values, 1)
next_month_index = monthly_revenue.index.max() + 1
forecast = slope * next_month_index + intercept

print("\nPREDICTIVE ANALYSIS")
print(f"Forecast revenue for month {next_month_index}: ${forecast:,.2f}")
print("(This is a simple linear-trend forecast, not a causal prediction.)")

# 4. Prescriptive analysis: identify the strongest segment and model a budget scenario
best_segment = segment_results.sort_values("marketing_roas", ascending=False).iloc[0]
best_key = segment_results["marketing_roas"].idxmax()
extra_budget = best_segment["marketing_spend"] * 0.10
estimated_extra_revenue = extra_budget * best_segment["marketing_roas"]

print("\nPRESCRIPTIVE ANALYSIS")
print(f"Top segment by historical ROAS: {best_key[0]} / {best_key[1]}")
print(f"Scenario: add 10% to its historical marketing spend (${extra_budget:,.2f}).")
print(f"Estimated additional revenue: ${estimated_extra_revenue:,.2f}")
print("(Scenario assumes historical ROAS remains constant; it is not a causal estimate.)")

DESCRIPTIVE ANALYSIS
        units  revenue   profit  marketing_spend
count  144.00   144.00   144.00           144.00
mean   132.41  4727.84  1787.74           267.50
std     23.73  1473.01   670.54            36.73
min     78.00  2223.00   752.00           184.00
25%    114.00  3501.00  1183.15           240.00
50%    132.00  4544.10  1615.80           267.50
75%    150.00  5984.55  2386.01           295.00
max    185.00  7742.25  3090.25           351.00

Revenue and profit by region:
          revenue    profit
region                     
West    244808.55  91201.35
South   227066.40  85950.30
North   208933.50  80282.40

DIAGNOSTIC ANALYSIS
                  revenue    profit  marketing_spend  units  profit_margin  \
region product                                                               
North  Basic     76237.50  26580.00             5520   2675           0.35   
       Premium  132696.00  53702.40             6360   3104           0.40   
South  Basic     83500.20  28643.7